<a href="https://colab.research.google.com/github/Banking-Analytics-Lab/DLinBankingBook/blob/main/Labs/TextBook_Lab_Foundation_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Foundation Models for Credit: Tables and Transaction Histories

*A new lab (2026), beyond the chapters of the book. It builds on the Chapter 3 lab (the data, the split and the recurrent models) and the Chapter 7 lab (fusion). About 1.5 hours on a Colab T4 GPU (Runtime → Change runtime type → T4 GPU).*

Foundation models are reaching banking in two forms: **tabular foundation models (TFMs)**, which predict from a table of examples given as context, and **transaction foundation models**, which turn a customer's history of events into an embedding. In this lab we build a small version of each, use a released TFM, and combine them on the Freddie Mac loans of the Chapter 3 lab.

You will:

1. **Pretrain a tiny TFM (NanoDPT)** on synthetic tables, see where its knowledge comes from, and then load a version we pretrained for 2.5 hours on a cluster GPU.
2. **Score loans without fitting**: give a TFM a context of labelled loans and score new ones. Compare it with XGBoost and with **TabDPT 1.3**, a released TFM.
3. **Pretrain a small PRAGMA-style encoder** on the monthly loan histories with masked modelling, and fine-tune it.
4. **Combine the two**: put the PRAGMA output into the TFM, following two of the fusion patterns Revolut described at GTC Berlin 2026.
5. **Validate the context**, as a model validator would: how much do the scores move when the context changes?

Every model is scored on the same 10,000 test loans as in Chapter 3, on all loans and on the loans that were current in March 2024.

**A note on terms.** Every model here gives a **score** that ranks loans. A score is not a probability of default (PD): a PD model must be calibrated to the portfolio, which we do not do here (see Exercise 4).

## 0. Setup

We install one package, `tabdpt` (Apache-2.0 licence), the released TabDPT model. Its weights also allow commercial use, so a bank could reuse this code. We do not use TabPFN-3.5 here: its weights are licensed for non-commercial use only. Check the licence of the weights, not only of the code, before a model goes into production.

In [ ]:
!pip install -q tabdpt==1.3.2

In [ ]:
import copy
import math
import os
import time
import warnings

# TabDPT's weights are public: do not let the Hugging Face library ask Colab for an HF_TOKEN secret.
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from xgboost import XGBClassifier
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=UserWarning)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
LAB_START = time.time()

## 1. Data and baselines

We use the data of the Chapter 3 lab, with the same preprocessing and the same split. In short:

- 50,000 mortgages, observed monthly from November 2021 to March 2024 (29 months × 8 variables), plus static variables recorded at origination.
- **Target:** delinquent at least once in April to June 2024 (2.4% of loans).
- **Features for the tabular models:** the static variables (credit score, CLTV, DTI and others) and hand-made **window** summaries of the monthly series (last value, means of the last 3 and 12 months, minimum and maximum of the last 12).
- **Split:** 40,000 training loans and 10,000 test loans, stratified, `random_state=42`.

The Chapter 3 lab explains each step. Here the code is condensed into one cell.

In [ ]:
!gdown -q 18ERhjQQg-PB6a5wEyUTNecJv8zkz0GkK
!gdown -q 1B2R1nCGIJ4TURvh7RL-aoLHAe8oP-uxO

df = pd.read_csv("Time_Series_Lab_sampled.csv", dtype={"CURRENT_LOAN_DELINQUENCY_STATUS": str})
static_df = pd.read_csv("TIme_Series_Lab_Origin_sampled.csv")
df["REPORTING_PERIOD"] = pd.to_datetime(df["REPORTING_PERIOD"].astype(str), format="%Y%m")
df = df.sort_values(["LOAN_NUMBER", "REPORTING_PERIOD"]).reset_index(drop=True)
status = pd.to_numeric(df["CURRENT_LOAN_DELINQUENCY_STATUS"], errors="coerce").fillna(1)   # "RA" -> 1
df["DELINQUENT"] = (status > 0).astype(int)

TARGET_START = pd.Timestamp("2024-04-01")
target = df[df.REPORTING_PERIOD >= TARGET_START].groupby("LOAN_NUMBER")["DELINQUENT"].max()
obs = df[df.REPORTING_PERIOD < TARGET_START].copy()          # the observation window only

SERIES = ["CURRENT_ACTUAL_UPB", "LOAN_AGE", "REMAINING_MONTHS", "CURRENT_INTEREST_RATE",
          "CURRENT_NON_INTEREST_BEARING_UPB", "ESTIMATED_LOAN_TO_VALUE", "INTEREST_BEARING_UPB", "DELINQUENT"]
obs["ESTIMATED_LOAN_TO_VALUE"] = obs["ESTIMATED_LOAN_TO_VALUE"].replace(999, np.nan)        # 999: not available
obs["ESTIMATED_LOAN_TO_VALUE"] = obs.groupby("LOAN_NUMBER")["ESTIMATED_LOAN_TO_VALUE"].ffill()
obs["ESTIMATED_LOAN_TO_VALUE"] = obs.groupby("LOAN_NUMBER")["ESTIMATED_LOAN_TO_VALUE"].bfill()

loans = np.sort(obs.LOAN_NUMBER.unique())
T = int(obs.groupby("LOAN_NUMBER").size().iloc[0])
X_raw = obs[SERIES].to_numpy(dtype=np.float32).reshape(len(loans), T, len(SERIES))   # (loans, months, variables)
y = target.reindex(loans).to_numpy(dtype=np.float32)

idx_train, idx_test = train_test_split(np.arange(len(loans)), test_size=0.2, stratify=y, random_state=42)
ltv = SERIES.index("ESTIMATED_LOAN_TO_VALUE")
X_raw[:, :, ltv] = np.where(np.isnan(X_raw[:, :, ltv]), np.nanmedian(X_raw[idx_train, :, ltv]), X_raw[:, :, ltv])

# Window summaries (40 features)
W = np.concatenate([X_raw[:, -1, :], X_raw[:, -3:, :].mean(1), X_raw[:, -12:, :].mean(1),
                    X_raw[:, -12:, :].min(1), X_raw[:, -12:, :].max(1)], axis=1)
dq = SERIES.index("DELINQUENT")
current = X_raw[:, -1, dq] == 0                             # loans not delinquent in March 2024

# Static variables (16 features)
st = static_df.set_index("LOAN_NUMBER").reindex(loans)
NUMERIC = ["CREDIT_SCORE", "CLTV", "DTI_RATIO", "ORIGINAL_UPB", "ORIGINAL_LOAN_TERM", "NUMBER_OF_BORROWERS"]
CATEGORICAL = ["FIRST_TIME_HOMEBUYER", "OCCUPANCY_STATUS", "PROPERTY_TYPE"]
num = st[NUMERIC].astype(float)
num["CREDIT_SCORE"] = num["CREDIT_SCORE"].replace(9999, np.nan)
num["DTI_RATIO"] = num["DTI_RATIO"].replace(999, np.nan)
num = num.fillna(num.iloc[idx_train].median())
scaler = StandardScaler().fit(num.iloc[idx_train])
encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore").fit(st[CATEGORICAL].iloc[idx_train])
S = np.concatenate([scaler.transform(num), encoder.transform(st[CATEGORICAL])], axis=1).astype(np.float32)
WS = np.concatenate([W, S], axis=1)

print(f"Panel {X_raw.shape}; positives {y.mean():.2%}; train {len(idx_train):,}, test {len(idx_test):,}")
print(f"Features: windows {W.shape[1]}, static {S.shape[1]}, both {WS.shape[1]}")

One function stores and prints the test AUC of every model, on all test loans and on the loans current in March 2024. We start with the baselines of Chapter 3.

In [ ]:
results = {}
yt, cur = y[idx_test], current[idx_test]


def report(name, scores):
    results[name] = (roc_auc_score(yt, scores), roc_auc_score(yt[cur], scores[cur]))
    print(f"{name:52s} AUC all {results[name][0]:.3f}   current {results[name][1]:.3f}")


def xgboost_scores(F_, rows):
    m = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=5, tree_method="hist", random_state=0)
    m.fit(F_[rows], y[rows])
    return m.predict_proba(F_[idx_test])[:, 1]


report("Rule: months delinquent", X_raw[idx_test, :, dq].sum(1))
report("XGBoost (static)", xgboost_scores(S, idx_train))
report("XGBoost (windows + static)", xgboost_scores(WS, idx_train))
logit = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced"))
logit.fit(WS[idx_train], y[idx_train])
report("Logistic regression (windows + static)", logit.predict_proba(WS[idx_test])[:, 1])

These models were **fitted** on the 40,000 training loans. Keep the numbers in mind: in Section 2 a model will score the same test loans without fitting anything.

## 2. NanoDPT: where a TFM's knowledge comes from

### 2.1 The model

NanoDPT is a small TFM in the style of TabDPT (Ma et al., 2025). Its input is one table: some **context rows** with their labels, and some **query rows** without labels.

1. Every row becomes one vector: a linear layer reads its standardized values and an indicator of which values are missing. The statistics for standardizing come from the context rows only.
2. The label of each context row is added to its vector, as a learned embedding.
3. In each of the 6 attention layers, **every row reads the context rows**. Query rows never read each other, so the score of one loan does not depend on which other loans are scored with it.
4. A final layer gives class logits for the query rows.

The model has 3.2 million parameters. TabDPT 1.3 has about 63 million (its weights file holds 252 MB of float32 values).

In [ ]:
N_FEAT, N_CLS, CLIP = 64, 10, 8.0     # features read (fewer are padded), largest number of classes, clip at +-8 sd


def standardize(x, n_ctx):
    """x (B, N, F) with NaN for missing. Standardize with the context rows only; return values and missing flags."""
    ctx = x[:, :n_ctx]
    mu = torch.nanmean(ctx, dim=1, keepdim=True)
    var = torch.nanmean((ctx - mu) ** 2, dim=1, keepdim=True)
    sd = torch.sqrt(torch.nan_to_num(var, nan=1.0)).clamp_min(1e-6)
    z = ((x - torch.nan_to_num(mu)) / sd).clamp(-CLIP, CLIP)
    miss = torch.isnan(z)
    return torch.nan_to_num(z, nan=0.0), miss.float()


class Block(nn.Module):
    """One layer: every row (context and query) attends to the context rows, then an MLP."""
    def __init__(self, d, heads, ff):
        super().__init__()
        self.n1, self.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.q, self.k, self.v, self.o = (nn.Linear(d, d) for _ in range(4))
        self.ff = nn.Sequential(nn.Linear(d, ff), nn.GELU(), nn.Linear(ff, d))
        self.heads = heads

    def forward(self, h, n_ctx):
        B, N, d = h.shape
        a, H = self.n1(h), self.heads
        q = self.q(a).view(B, N, H, d // H).transpose(1, 2)
        k = self.k(a[:, :n_ctx]).view(B, n_ctx, H, d // H).transpose(1, 2)     # keys and values: context rows only
        v = self.v(a[:, :n_ctx]).view(B, n_ctx, H, d // H).transpose(1, 2)
        att = F.scaled_dot_product_attention(q, k, v)
        h = h + self.o(att.transpose(1, 2).reshape(B, N, d))
        return h + self.ff(self.n2(h))


class NanoDPT(nn.Module):
    def __init__(self, d=256, layers=6, heads=4, ff=512, n_feat=N_FEAT, n_cls=N_CLS):
        super().__init__()
        self.config = dict(d=d, layers=layers, heads=heads, ff=ff, n_feat=n_feat, n_cls=n_cls)
        self.enc = nn.Linear(2 * n_feat, d)                       # values and missing flags
        self.y_emb = nn.Embedding(n_cls, d)                       # label embedding, added to context rows
        self.blocks = nn.ModuleList(Block(d, heads, ff) for _ in range(layers))
        self.norm = nn.LayerNorm(d)
        self.head = nn.Linear(d, n_cls)

    def forward(self, x, y_ctx):
        """x (B, N, F): the first n_ctx rows are the context. y_ctx (B, n_ctx). Returns logits for the query rows."""
        n_ctx = y_ctx.shape[1]
        z, miss = standardize(x, n_ctx)
        pad = self.config["n_feat"] - z.shape[-1]
        z, miss = F.pad(z, (0, pad)), F.pad(miss, (0, pad))
        h = self.enc(torch.cat([z, miss], dim=-1))
        h = torch.cat([h[:, :n_ctx] + self.y_emb(y_ctx), h[:, n_ctx:]], dim=1)
        for blk in self.blocks:
            h = blk(h, n_ctx)
        return self.head(self.norm(h[:, n_ctx:]))

    @torch.no_grad()
    def predict_proba(self, X_ctx, y_ctx, X_qry, n_classes=2, chunk=2048):
        """Like scikit-learn: numpy in, class probabilities for the query rows out. No fitting."""
        self.eval()
        dev = next(self.parameters()).device
        Xc = torch.as_tensor(X_ctx, dtype=torch.float32, device=dev)[None]
        yc = torch.as_tensor(y_ctx, dtype=torch.long, device=dev)[None]
        out = []
        for i in range(0, len(X_qry), chunk):
            Xq = torch.as_tensor(X_qry[i:i + chunk], dtype=torch.float32, device=dev)[None]
            out.append(torch.softmax(self(torch.cat([Xc, Xq], 1), yc)[0, :, :n_classes].float(), -1).cpu())
        return torch.cat(out).numpy()

### 2.2 The prior: tables that do not exist

A TFM learns from **many** tables, not one. Like TabPFN (Hollmann et al., 2023), NanoDPT is pretrained on synthetic tables drawn from a **prior**, a recipe that generates tables:

1. Draw a random structural causal model: a random network of 1 to 4 layers with random weights, sparse links, random activations and noise. Its hidden units are the variables.
2. Pass random inputs through it. Pick some units as features and one unit as the target.
3. Cut the target into classes. Because credit data are unbalanced, a binary target gets a random positive rate between 1% and 50%.
4. Make some features categorical, some log-normal, and some partly missing.

Each table is used for one training step and then thrown away. What remains in the weights is a **way to predict from a context**, not any particular table.

In [ ]:
ACTS = [torch.tanh, torch.relu, torch.sin, lambda t: torch.sign(t), lambda t: t ** 2 / 3,
        lambda t: torch.sigmoid(3 * t), lambda t: t, lambda t: torch.abs(t)]


def one_table(n, gen, dev):
    r = lambda lo, hi: int(torch.randint(lo, hi + 1, (1,), generator=gen))
    u = lambda: float(torch.rand(1, generator=gen))
    layers, width = r(1, 4), r(8, 48)
    noise = 10 ** (-2 + 1.5 * u())
    z = torch.randn(n, width, device=dev) * (1 if u() < 0.5 else 0.5 + u())
    nodes = []
    for _ in range(layers):
        Wm = torch.randn(z.shape[1], width, device=dev) / math.sqrt(z.shape[1])
        Wm = Wm * (torch.rand(z.shape[1], width, device=dev) < 0.2 + 0.8 * u())          # sparse links
        z = ACTS[r(0, len(ACTS) - 1)](z @ Wm + 0.5 * torch.randn(1, width, device=dev))
        z = z + noise * torch.randn_like(z)
        nodes.append(z)
    V = torch.cat(nodes, dim=1)
    V = V[:, V.std(0) > 1e-4]
    if V.shape[1] < 3:
        return None
    perm = torch.randperm(V.shape[1], device=dev)
    t = V[:, perm[0]]                                              # the target node
    f = min(r(2, N_FEAT), V.shape[1] - 1)
    X = V[:, perm[1:f + 1]].clone()
    if u() < 0.6:                                                  # binary, positive rate 1% to 50%
        rate = 10 ** (-2 + math.log10(50) * u())
        yy, k = (t > torch.quantile(t, 1 - rate)).long(), 2
    else:                                                          # 3 to 10 classes
        cuts = torch.quantile(t, torch.sort(torch.rand(r(3, N_CLS) - 1, device=dev)).values * 0.9 + 0.05)
        yy = torch.bucketize(t, cuts)
        k = int(yy.max()) + 1
    if yy.max() == 0:
        return None
    for j in range(X.shape[1]):                                    # categorical, log-normal, missing
        c = u()
        if c < 0.15:
            X[:, j] = torch.bucketize(X[:, j], torch.quantile(X[:, j], torch.linspace(0, 1, r(3, 12), device=dev)[1:-1])).float()
        elif c < 0.25:
            X[:, j] = torch.exp(X[:, j].clamp(-5, 5))
        if u() < 0.1:
            X[torch.rand(n, device=dev) < 0.3 * u(), j] = float("nan")
    return X, yy, k


def sample_batch(B, n, gen, dev):
    xs, ys, ks = [], [], []
    while len(xs) < B:
        t = one_table(n, gen, dev)
        if t is None:
            continue
        X, yy, k = t
        xs.append(F.pad(X, (0, N_FEAT - X.shape[1]))), ys.append(yy), ks.append(k)
    return torch.stack(xs), torch.stack(ys), torch.tensor(ks, device=dev)


def episode_loss(model, x, yy, k, n_ctx):
    """Cross-entropy on the query rows of each synthetic table; classes beyond the table's k are masked."""
    logits = model(x, yy[:, :n_ctx])
    mask = torch.arange(N_CLS, device=x.device)[None, None, :] >= k[:, None, None]
    return F.cross_entropy(logits.float().masked_fill(mask, -1e4).reshape(-1, N_CLS), yy[:, n_ctx:].reshape(-1))

### 2.3 Pretrain for two minutes

We pretrain a fresh NanoDPT for two minutes: batches of 16 synthetic tables with 512 rows, 384 of them as context. The loss is the cross-entropy on the hidden query labels. Watch it fall: the model is learning to read a context.

In [ ]:
torch.manual_seed(0)
gen = torch.Generator().manual_seed(0)
fresh = NanoDPT().to(device)
opt = torch.optim.AdamW(fresh.parameters(), lr=3e-4, weight_decay=0.01)
scaler16 = torch.amp.GradScaler(enabled=device.type == "cuda")     # half precision needs loss scaling
losses, t0 = [], time.time()
while time.time() - t0 < 120:                                     # two minutes
    x, yy, k = sample_batch(16, 512, gen, device)
    with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
        loss = episode_loss(fresh, x, yy, k, 384)
    opt.zero_grad()
    scaler16.scale(loss).backward()
    scaler16.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(fresh.parameters(), 1.0)
    scaler16.step(opt)
    scaler16.update()
    losses.append(loss.item())
print(f"{len(losses)} steps in {time.time() - t0:.0f} s")
plt.figure(figsize=(7, 3))
plt.plot(pd.Series(losses).rolling(25).mean())
plt.xlabel("step"); plt.ylabel("query loss (moving average)"); plt.title("NanoDPT pretraining on synthetic tables")
plt.show()

Two minutes are not enough for a useful model. We pretrained the same NanoDPT for **150 minutes** on a 3g.40gb slice of an H100 GPU (Alliance cluster Nibi): 140,274 steps of 32 tables with 1,024 rows each. On synthetic tables it never saw, it matches a logistic regression fitted to each table (mean AUC 0.780 against 0.779 over 204 binary tables), without fitting anything. We load that checkpoint.

In [ ]:
!gdown -q 1TMOUkivcfRrdx-wukvwxC5rhzuZvxzG- -O nanodpt.pt
ck = torch.load("nanodpt.pt", map_location="cpu", weights_only=False)
nano = NanoDPT(**ck["config"]).to(device)
nano.load_state_dict(ck["state_dict"])
nano.eval()
print(f"NanoDPT checkpoint: {ck['step']:,} pretraining steps, {sum(p.numel() for p in nano.parameters()):,} parameters")

### 2.4 Scoring loans without fitting

Now the context is a sample of **training loans** with their labels, and the queries are the test loans. NanoDPT was pretrained on contexts of 768 rows, so we use contexts of about that size.

How should we choose the context rows? A random sample of 1,024 loans contains only about 24 delinquent ones. We compare it with a **balanced** context, in which a quarter of the rows are delinquent loans. The context is the TFM's training data, so this is the same choice as undersampling a training set.

In [ ]:
pos_train, neg_train = idx_train[y[idx_train] == 1], idx_train[y[idx_train] == 0]


def draw_context(n, balanced, seed, pool=None):
    """Indices of n context rows from the training loans (or another pool): random, or a quarter positives."""
    pool = idx_train if pool is None else pool
    r = np.random.default_rng(seed)
    if not balanced:
        return r.choice(pool, n, replace=False)
    pos, neg = pool[y[pool] == 1], pool[y[pool] == 0]
    k = min(n // 4, len(pos))
    return np.concatenate([r.choice(pos, k, replace=False), r.choice(neg, n - k, replace=False)])


def nano_scores(F_, ctx, rows=None):
    rows = idx_test if rows is None else rows
    return nano.predict_proba(F_[ctx], y[ctx].astype(int), F_[rows])[:, 1]


t0 = time.time()
for name, F_ in [("static", S), ("windows + static", WS)]:
    report(f"NanoDPT ({name}, random context 1,024)", nano_scores(F_, draw_context(1024, False, 0)))
    report(f"NanoDPT ({name}, balanced context 1,024)", nano_scores(F_, draw_context(1024, True, 0)))
print(f"Four runs, 10,000 test loans each, in {time.time() - t0:.1f} s")

With a balanced context of 1,024 loans, NanoDPT scores the test loans about as well as XGBoost fitted on all 40,000 training loans. It never trained on a mortgage: everything it knows about reading a table comes from synthetic tables.

**Question.** Why does the balanced context help more on the loans that are current at the cut-off?

## 3. A released TFM: TabDPT 1.3

TabDPT (Ma et al., 2025), from Layer 6, TD Bank's AI lab, is pretrained on real tables, not only synthetic ones, with a random column taken as the target in each step. It is about 20 times larger than NanoDPT. The first call downloads its weights (252 MB) from Hugging Face.

`fit` only stores the training rows: nothing is trained. At prediction time, TabDPT builds a context of `context_size` training rows for the test rows. We compare a random subsample with a balanced one. `batch_size` keeps memory low, because every test loan carries its own copy of the context. `use_flash=False` is needed on a T4: TabDPT asks PyTorch for a flash-attention kernel, which the T4 does not have.

In [ ]:
from tabdpt import TabDPTClassifier

tabdpt = TabDPTClassifier(compile=False, use_flash=False, verbose=False)   # T4: no flash-attention kernel
tabdpt.fit(WS[idx_train], y[idx_train].astype(int))
for mode in ["subsample", "subsample-balanced"]:
    tabdpt.context_reduction = mode
    t0 = time.time()
    p = tabdpt.ensemble_predict_proba(WS[idx_test], n_ensembles=1, context_size=1024, batch_size=256, seed=0)[:, 1]
    report(f"TabDPT (windows + static, {mode} 1,024)", p)
    print(f"   {time.time() - t0:.1f} s")

TabDPT has a third option, **retrieval**, which gives every test loan its own context: its 1,024 nearest training loans. We do not run it here, because on a T4 it takes more than 6 minutes for the 10,000 test loans. In our pilot on a cluster GPU it gave AUC 0.879 overall and 0.773 on current loans, no better than a random context, and took about 70 times longer than the balanced one.

On this data, TabDPT with a balanced context is slightly ahead of XGBoost and NanoDPT. Differences of 0.01 in AUC on one split may be noise; Section 6 shows how much the AUC moves when only the context changes.

## 4. A small PRAGMA: an encoder for event histories

PRAGMA (Ostroukhov et al., 2026) is Revolut's foundation model for customer events. Its three main ideas fit in a few lines:

1. **Key–value–time tokens.** Each event becomes tokens: a *key* (which field), a *value* (the field's value as a percentile bucket or a category), and the *time* of the event.
2. **A bidirectional transformer encoder** reads all tokens of a customer at once.
3. **Masked modelling.** Some tokens are hidden, and the model learns to predict them. No labels are needed, so a bank can pretrain on all its customers.

Our events are the 29 monthly records of each loan, and each record has 8 keys: 232 tokens per loan. Values become 16 percentile buckets per variable, fitted on the training loans (the delinquency flag keeps its two values).

In [ ]:
N_BUCKETS = 16
MASK, MISSING = N_BUCKETS, N_BUCKETS + 1


class Tokenizer:
    def fit(self, X, rows):
        self.edges = []
        for j in range(X.shape[2]):
            v = X[rows, :, j].ravel()
            u = np.unique(v[~np.isnan(v)])
            if len(u) <= N_BUCKETS:                    # few values (the delinquency flag): one bucket each
                e = (u[1:] + u[:-1]) / 2
            else:
                e = np.unique(np.quantile(v[~np.isnan(v)], np.linspace(0, 1, N_BUCKETS + 1)[1:-1]))   # percentiles
            self.edges.append(e)
        return self

    def transform(self, X):
        out = np.empty(X.shape, dtype=np.int64)
        for j, e in enumerate(self.edges):
            out[:, :, j] = np.where(np.isnan(X[:, :, j]), MISSING, np.searchsorted(e, X[:, :, j], side="right"))
        return out


tok = Tokenizer().fit(X_raw, idx_train).transform(X_raw)
print("Token array:", tok.shape, "  buckets per key:", [len(e) + 1 for e in Tokenizer().fit(X_raw, idx_train).edges])
print("Loan 0, last three months:", tok[0, -3:].tolist())

**What to mask.** In a first pilot we hid single tokens. The task was too easy: a balance in one month is almost the same as in the month before, so the model copied it from its neighbour, and the loss fell to almost zero. Here we hide **whole months**: two runs of three consecutive months per loan, all 8 keys at once. The model must now fill a gap from the months around it, not copy from the same month.

In [ ]:
class SmallPragma(nn.Module):
    def __init__(self, n_keys=8, n_months=29, d=64, layers=4, heads=4, ff=128, dropout=0.1):
        super().__init__()
        self.n_keys, self.n_months = n_keys, n_months
        self.key = nn.Embedding(n_keys, d)
        self.val = nn.Embedding(n_keys * (N_BUCKETS + 2), d)          # one value table per key
        self.time = nn.Embedding(n_months, d)
        layer = nn.TransformerEncoderLayer(d, heads, ff, dropout, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d)
        self.mlm = nn.Linear(d, n_keys * N_BUCKETS)                  # predicts the bucket of a hidden token
        self.head = nn.Linear(d, 1)                                  # for fine-tuning
        self.register_buffer("keys", torch.arange(n_keys).repeat(n_months), persistent=False)
        self.register_buffer("months", torch.arange(n_months).repeat_interleave(n_keys), persistent=False)

    def encode(self, t):                                             # t (B, months, keys) -> (B, months*keys, d)
        v = t.reshape(t.shape[0], -1) + self.keys * (N_BUCKETS + 2)
        return self.norm(self.encoder(self.key(self.keys) + self.val(v) + self.time(self.months)))

    def embed(self, t):
        return self.encode(t).mean(1)                                # the loan embedding

    def logit(self, t):
        return self.head(self.embed(t)).squeeze(-1)

    def masked_loss(self, t, spans=2, span_len=3):
        B, M, K = t.shape
        starts = torch.randint(0, M - span_len + 1, (B, spans), device=t.device)
        m = torch.arange(M, device=t.device)
        hide_month = ((m >= starts[..., None]) & (m < starts[..., None] + span_len)).any(1)   # (B, M)
        flat = t.reshape(B, -1)
        hide = hide_month.repeat_interleave(K, dim=1) & (flat < N_BUCKETS)
        h = self.encode(torch.where(hide, torch.full_like(flat, MASK), flat).reshape(t.shape))[hide]
        k = self.keys.expand(B, -1)[hide]
        logits = self.mlm(h).view(-1, K, N_BUCKETS)[torch.arange(len(k)), k]
        return F.cross_entropy(logits, flat[hide])


@torch.no_grad()
def pragma_apply(model, rows, what="embed", bs=1024):
    model.eval()
    T_ = torch.as_tensor(tok[rows])
    out = []
    for i in range(0, len(rows), bs):
        t = T_[i:i + bs].to(device)
        out.append((model.embed(t) if what == "embed" else torch.sigmoid(model.logit(t))).cpu())
    return torch.cat(out).numpy()

### 4.1 Pretraining

We pretrain on the **training loans only**, without their labels, for 10 epochs. A bank would pretrain on every customer it has, labelled or not; we keep the test loans out so that the comparison stays clean.

In [ ]:
torch.manual_seed(0)
pragma = SmallPragma().to(device)
print(f"SmallPragma: {sum(p.numel() for p in pragma.parameters()):,} parameters")
EPOCHS, BS = 10, 256
opt = torch.optim.AdamW(pragma.parameters(), lr=1e-3, weight_decay=0.01)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, 1e-3, total_steps=EPOCHS * math.ceil(len(idx_train) / BS), pct_start=0.1)
T_train = torch.as_tensor(tok[idx_train])
t0 = time.time()
for ep in range(EPOCHS):
    pragma.train()
    tot = 0.0
    for b in torch.randperm(len(T_train)).split(BS):
        loss = pragma.masked_loss(T_train[b].to(device))
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        tot += loss.item() * len(b)
    print(f"epoch {ep + 1:2d}: masked-month loss {tot / len(T_train):.3f}   (guessing: {math.log(N_BUCKETS):.3f})   {time.time() - t0:.0f} s")

The loss still falls far below guessing. In our test run it ended at 0.13, against 0.09 when single tokens were hidden. Hiding whole months makes the task harder, but not hard: the loan age and the remaining term follow from the neighbouring months, and balances change slowly. A harder task, such as predicting the next months instead of filling a gap, may teach the encoder more. That is a design choice to test, not a rule.

### 4.2 What the embeddings know

Every loan now has a 64-dimensional embedding. The encoder never saw a label. Does the embedding still separate the loans that become delinquent? A **linear probe** (a logistic regression on the embedding) answers that.

In [ ]:
E_base = pragma_apply(pragma, np.arange(len(loans)))
probe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, class_weight="balanced"))
probe.fit(E_base[idx_train], y[idx_train])
report("PRAGMA base embeddings, linear probe", probe.predict_proba(E_base[idx_test])[:, 1])
report("XGBoost (base embeddings + static)", xgboost_scores(np.concatenate([E_base, S], 1), idx_train))

### 4.3 Fine-tuning

PRAGMA adapts its encoder to each task by fine-tuning (with LoRA, on 2 to 4% of its parameters). Our encoder is small, so we fine-tune all of it. We split the training loans again, as in Chapter 3: two thirds to fine-tune (`idx_fit`), one third held out (`idx_val`) to choose the epoch. A head on the pooled embedding gives the score, and delinquent loans get a larger weight in the loss.

In [ ]:
idx_fit, idx_val = train_test_split(idx_train, test_size=0.33, stratify=y[idx_train], random_state=42)
pos_weight = torch.tensor((y[idx_fit] == 0).sum() / (y[idx_fit] == 1).sum(), device=device)
opt = torch.optim.AdamW(pragma.parameters(), lr=3e-4, weight_decay=0.01)
Y_ = torch.as_tensor(y)
best_auc, best_state = -1, None
for ep in range(3):
    pragma.train()
    for b in torch.as_tensor(idx_fit)[torch.randperm(len(idx_fit))].split(256):
        loss = F.binary_cross_entropy_with_logits(pragma.logit(torch.as_tensor(tok[b]).to(device)),
                                                  Y_[b].to(device), pos_weight=pos_weight)
        opt.zero_grad(); loss.backward(); opt.step()
    auc = roc_auc_score(y[idx_val], pragma_apply(pragma, idx_val, "score"))
    print(f"fine-tune epoch {ep + 1}: validation AUC {auc:.4f}")
    if auc > best_auc:
        best_auc, best_state = auc, copy.deepcopy(pragma.state_dict())
pragma.load_state_dict(best_state)
report("PRAGMA fine-tuned (its own head)", pragma_apply(pragma, idx_test, "score"))

## 5. PRAGMA inside the TFM

At GTC Berlin 2026, Revolut described four ways to combine PRAGMA with Strider, its tabular foundation model. We test two of them:

- **Pattern 2, embeddings as columns.** The fine-tuned embedding, compressed to 16 principal components, becomes 16 extra columns next to the static variables.
- **Pattern 3, score as a feature.** The fine-tuned PRAGMA score becomes one extra column.

**One rule matters here.** The fine-tuned model has seen the labels of the `idx_fit` loans, so its scores on those loans are too good. If those loans were the TFM's context, the TFM would learn to trust the score too much. So the context, and the training rows of XGBoost, come from `idx_val` only: loans the fine-tuning did not train on. This is the same reason stacking uses out-of-fold predictions. (`idx_val` also chose the fine-tuning epoch, a small leak we accept here.)

We give the windows to none of these models: PRAGMA must replace the hand-made summaries.

In [ ]:
E_ft = pragma_apply(pragma, np.arange(len(loans)))
score_ft = pragma_apply(pragma, np.arange(len(loans)), "score")[:, None].astype(np.float32)
pca = PCA(16, random_state=0).fit(E_ft[idx_val])
P_emb = np.concatenate([pca.transform(E_ft), S], 1).astype(np.float32)      # pattern 2
P_score = np.concatenate([score_ft, S], 1)                                  # pattern 3

report("XGBoost (static), idx_val rows", xgboost_scores(S, idx_val))
report("XGBoost (PRAGMA emb + static), idx_val rows", xgboost_scores(P_emb, idx_val))
report("XGBoost (PRAGMA score + static), idx_val rows", xgboost_scores(P_score, idx_val))
for name, F_ in [("PRAGMA emb + static", P_emb), ("PRAGMA score + static", P_score)]:
    report(f"NanoDPT ({name}), balanced 1,024 from idx_val", nano_scores(F_, draw_context(1024, True, 0, idx_val)))
tabdpt_score = TabDPTClassifier(compile=False, use_flash=False, verbose=False, context_reduction="subsample-balanced")
tabdpt_score.fit(P_score[idx_val], y[idx_val].astype(int))
report("TabDPT (PRAGMA score + static), balanced 1,024",
       tabdpt_score.ensemble_predict_proba(P_score[idx_test], n_ensembles=1, context_size=1024, batch_size=256, seed=0)[:, 1])

**Compare** these rows with "XGBoost (windows + static)" and with the hybrid GRU of Chapter 3 (AUC 0.905 on all test loans, 0.826 on current loans). Did PRAGMA learn summaries as useful as the hand-made windows? Did the TFM gain from it? Revolut reported large gains on its own data (Ostroukhov et al., 2026). Here the history is only 8 variables over 29 months, and the windows already capture much of it.

In our test run, TabDPT with the PRAGMA score and the static variables reached AUC 0.905 on all loans and 0.825 on current loans, level with the hybrid GRU, and without the hand-made windows. Before you call that a result, look at Section 6.1: one context draw is one sample, and the AUC moves when the context changes.

## 6. Validating the context

For a fitted model, a validator checks the training data and the fitted parameters. For a TFM, the weights are fixed and the **context is part of the model**: change it, and the scores change. A validator must ask how much. We use NanoDPT with windows and static variables, because it is fast enough to run many contexts.

### 6.1 The same model, different contexts

In [ ]:
rows = []
for balanced in (False, True):
    for seed in range(20):
        p = nano_scores(WS, draw_context(1024, balanced, seed))
        rows.append({"context": "balanced" if balanced else "random", "seed": seed,
                     "AUC all": roc_auc_score(yt, p), "AUC current": roc_auc_score(yt[cur], p[cur]), "scores": p})
draws = pd.DataFrame(rows)
print(draws.groupby("context")[["AUC all", "AUC current"]].agg(["mean", "std", "min", "max"]).round(3))
draws.boxplot(column="AUC all", by="context", figsize=(6, 3.5))
plt.suptitle(""); plt.title("NanoDPT, 20 context draws of 1,024 loans each"); plt.ylabel("test AUC"); plt.show()

### 6.2 Individual loans

A portfolio AUC can be stable while individual scores move. Suppose a bank reviews the riskiest 5% of loans. How often does a loan enter or leave that list when only the context changes?

In [ ]:
bal = np.stack(draws[draws.context == "balanced"].scores.values)       # (20 draws, 10,000 test loans)
flag = bal >= np.quantile(bal, 0.95, axis=1, keepdims=True)              # riskiest 5% in each draw
share = flag.mean(0)
print(f"Loans flagged in at least one draw: {(share > 0).sum():,}; in all 20 draws: {(share == 1).sum():,} (each list has {flag[0].sum():,})")
print(f"Loans flagged in some draws but not others: {((share > 0) & (share < 1)).sum():,}")
rank_sd = pd.DataFrame(bal.T).rank(pct=True).std(1)
print(f"Median standard deviation of a loan's percentile rank across draws: {rank_sd.median():.3f}")

### 6.3 Context size

In [ ]:
size_rows = []
for n in [128, 256, 512, 1024, 2048, 4096]:
    for seed in range(5):
        p = nano_scores(WS, draw_context(n, True, seed))
        size_rows.append({"context size": n, "AUC all": roc_auc_score(yt, p), "AUC current": roc_auc_score(yt[cur], p[cur])})
sizes = pd.DataFrame(size_rows).groupby("context size").agg(["mean", "std"]).round(3)
print(sizes)

NanoDPT was pretrained with contexts of 768 rows. Does it use a larger context well? A model pretrained on longer contexts, such as TabDPT, may behave differently.

### 6.4 Who is in the context

The context can leave out a group of borrowers. Here we score the **investor** loans (occupancy status `I`) twice: once with a balanced context drawn from all training loans, and once with a balanced context that holds **no** investor loans. Fairness and accuracy for a group depend on whether that group is in the context (Kenfack et al., 2026).

In [ ]:
occ = st["OCCUPANCY_STATUS"].to_numpy()
inv_test = idx_test[occ[idx_test] == "I"]
no_inv_pool = idx_train[occ[idx_train] != "I"]
print(f"Investor loans: {len(inv_test):,} in the test set, {y[inv_test].mean():.2%} delinquent")
for label, pool in [("all training loans", None), ("no investor loans", no_inv_pool)]:
    aucs, means = [], []
    for seed in range(10):
        p = nano_scores(WS, draw_context(1024, True, seed, pool), rows=inv_test)
        aucs.append(roc_auc_score(y[inv_test], p)); means.append(p.mean())
    print(f"Context from {label:20s}: investor AUC {np.mean(aucs):.3f} (sd {np.std(aucs):.3f}), mean score {np.mean(means):.3f}")

Only about 8 of the investor loans in the test set are delinquent, so their AUC is too noisy to compare. Look at the **mean score** instead. In our test run it roughly doubled, from 0.10 to 0.21, when the context held no investor loans: the same borrowers looked riskier only because no one like them was in the context. With a fixed cutoff, that changes who is declined.

**For a validator.** The context is a model input that changes the output, like a training set that is read at prediction time. A validation report for a TFM should state how the context is chosen, its size and composition, how it is versioned, and the spread of scores and decisions across context draws. Section 6.2 shows why the last point matters to the customer, not only to the AUC.

## 7. Results

In [ ]:
table = pd.DataFrame(results, index=["AUC all loans", "AUC current loans"]).T.round(3)
table.loc["Hybrid GRU (Chapter 3 lab, reported)"] = [0.905, 0.826]
print(table.to_string())
print(f"\nLab run time: {(time.time() - LAB_START) / 60:.1f} minutes")

Read the table with three questions:

1. **Fitting against context.** How close do the TFMs get to XGBoost, without fitting, and with how many labelled loans?
2. **Learned against hand-made features.** Do the PRAGMA embeddings or score replace the windows?
3. **Stability.** Compare the gaps between models with the spread across context draws in Section 6.1. Which gaps are larger than the noise?

**Our test run** (Colab T4, 20 November 2026; about 8 minutes of computing, so most of the 1.5 hours is reading and discussion). AUC on all test loans and on current loans:

| Model | All | Current |
|---|---|---|
| XGBoost (windows + static), fitted on 40,000 loans | 0.892 | 0.798 |
| NanoDPT (windows + static), balanced context of 1,024 | 0.894 | 0.802 |
| TabDPT (windows + static), balanced context of 1,024 | 0.901 | 0.814 |
| TabDPT (windows + static), random context of 1,024 | 0.881 | 0.776 |
| PRAGMA fine-tuned, its own head | 0.874 | 0.765 |
| NanoDPT (PRAGMA score + static) | 0.900 | 0.818 |
| TabDPT (PRAGMA score + static) | 0.905 | 0.825 |
| Hybrid GRU (Chapter 3 lab) | 0.905 | 0.826 |

Across 20 balanced context draws, NanoDPT's AUC ranged from 0.878 to 0.907 (random draws: 0.832 to 0.882). Most gaps in the table are smaller than that range. The clear findings are two: a TFM matches a fitted XGBoost without fitting, and the way the context is drawn matters more than the choice between the two TFMs.

## 8. Exercises

1. **Out of time.** All loans here come from the same period. How would you build an out-of-time test of a TFM, and what would its context be at prediction time?
2. **Another prior.** Change the prior of Section 2.2: for example, make every binary target have a positive rate below 5%, or add tree-like functions (Mitra mixes such priors). Pretrain for two minutes and compare with the original prior after the same time.
3. **PRAGMA without the delinquency key.** Remove the `DELINQUENT` key from the tokens, pretrain again, and compare the probe. How much did the encoder rely on past delinquency?
4. **From scores to PDs.** Calibrate the NanoDPT score to the portfolio: fit an isotonic or Platt calibration on held-out training loans, and compare the mean calibrated score with the observed delinquency rate (2.4%). Why does a balanced context make the raw scores too high?
5. **The other fusion patterns.** Try pattern 1 (concatenate the embedding and the static data into one network) or pattern 4 (average the TFM and PRAGMA scores). Which pattern would you defend to a validator, and why?
6. **Retrieval.** Run TabDPT with `context_reduction = "retrieval"` on 1,000 test loans. Time it, and compare its AUC with the balanced context on the same loans. When would a per-loan context be worth its cost?

## References

- Bravo, C., Maldonado, S. and Óskarsdóttir, M. (2026). *Deep Learning in Banking*. Wiley. Chapter 3 lab.
- Hollmann, N., Müller, S., Eggensperger, K. and Hutter, F. (2023). TabPFN: A transformer that solves small tabular classification problems in a second. ICLR.
- Kenfack, P. J., Ebrahimi Kahou, S. and Aïvodji, U. (2026). Towards fair in-context learning with tabular foundation models. TMLR.
- Ma, J. et al. (2025). TabDPT: Scaling tabular foundation models on real data. NeurIPS. arXiv:2410.18164.
- Müller, S., Hollmann, N., Pineda Arango, S., Grabocka, J. and Hutter, F. (2022). Transformers can do Bayesian inference. ICLR.
- Ostroukhov, M. et al. (2026). PRAGMA: Revolut foundation model. arXiv:2604.08649.
- Revolut (2026). GTC Berlin 2026, session SB61175 (NVIDIA on-demand): PRAGMA and Strider, fusion patterns.
- Xu, J., Zhou, T., Tai, T., Pan, J., Statnikov, A. and Kumar, N. (2026). An interactive guide to tabular foundation models. NeurIPS tutorial, https://affirm.github.io/tabular-foundation-models-tutorial/ (recommended preparation).